# 02 - Compare TF-IDF and BM25 over the full corpus

**Objective:** build two transparent lexical baselines and inspect their ranking differences.
**Setup:** prepare SciFact as described in the README. This notebook runs independently and offline after data preparation.

Both methods search all 5,183 documents. We index titles and abstracts. TF-IDF uses raw term counts, `log((N+1)/(df+1))+1`, and cosine normalization. BM25 uses term saturation and document length normalization, with fixed `k1=1.2` and `b=0.75`. Neither is a neural embedding model. We retain digits and negation; both use the same tokenizer without stemming or stopword removal.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "rag_lab/core.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the Awesome-RAG checkout")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from rag_lab.core import load_dataset, chunk_document, LexicalIndex, assemble_context
corpus, queries, qrels, manifest = load_dataset(ROOT)
print(f"{len(corpus):,} abstracts; {len(qrels['train']):,} development claims; {len(qrels['test']):,} test claims")
print("Snapshot:", manifest["archive_sha256"])


In [ ]:
from time import perf_counter
indexes = {}
for method in ["tfidf", "bm25"]:
    start = perf_counter()
    indexes[method] = LexicalIndex(list(corpus.values()), method=method)
    print(method, "vocabulary:", len(indexes[method].idf), "build seconds:", round(perf_counter() - start, 3))


## Read the scoring implementation

An inverted index accumulates scores only for documents sharing query terms. Stable document-ID tie breaks make rankings repeatable. The TF-IDF query uses corpus IDF values; test queries never contribute to fitting the index. Indexing the complete benchmark corpus is expected in this retrieval setting.


In [ ]:
import inspect
print(inspect.getsource(LexicalIndex))


In [ ]:
qid = sorted(qrels["train"])[0]
claim = queries[qid]["text"]
print("CLAIM:", claim)
print("LABELED SOURCES:", sorted(qrels["train"][qid]))
for method, index in indexes.items():
    print("\n", method)
    for rank, hit in enumerate(index.search(claim, 5), 1):
        print(rank, hit["id"], round(hit["score"], 4),
              "JUDGED RELEVANT" if hit["id"] in qrels["train"][qid] else "UNJUDGED", hit["title"])
assert all(index.search("") == [] for index in indexes.values())


## Find disagreements instead of choosing only successful examples

Inspect the first five rank-one disagreements on development claims. A score is not a probability, and BM25 and cosine scores have different scales. An unjudged document is not necessarily irrelevant; evaluation relies on the published judgments.


In [ ]:
disagreements = []
for qid in sorted(qrels["train"]):
    first = {name: index.search(queries[qid]["text"], 1) for name, index in indexes.items()}
    ids = {name: hits[0]["id"] if hits else None for name, hits in first.items()}
    if ids["tfidf"] != ids["bm25"]:
        disagreements.append((qid, ids))
print("Rank-one disagreements:", len(disagreements), "of", len(qrels["train"]))
for qid, ids in disagreements[:5]:
    print(qid, queries[qid]["text"], ids, "judged:", sorted(qrels["train"][qid]))


## Exercise

Try a different tokenizer using only the training claims. Record the exact change and measure it across all development queries before looking at test results. Scientific names, hyphenation, negation, and synonyms are useful failure categories.

Next: [03 - Evidence and generation](03_context_and_generation.ipynb).
